# Fault Lab 1

## Task 1

**SUMMARY:** *Microcontrollers and FPGAs have a number of operating conditions that must be met in order for the device to work properly. Outside of these conditions, devices will begin to malfunction, with more extreme violations causing the device to stop entirely or even become damaged. By going outside these operating conditions for very small amounts of time, we can cause a varitey of temporary malfunctions*

*In this lab, we'll explore clock glitching, which inserts short glitches into a device's clock. This will be used to get a target that's summing numbers in a loop to arrive at the wrong result.*

**LEARNING OUTCOMES:**

* Understand effects of clock glitching
* Exploring ChipWhisperer's glitch module
* Using clock glitching to disrupt a target's algorithm

## Clock Glitching Theory

Digital hardware devices almost always expect some form of reliable clock. We can manipulate the clock being presented to the device to cause unintended behaviour. We'll be concentrating on microcontrollers here, however other digital devices (e.g. hardware encryption accelerators) can also have faults injected using this technique.

Consider a microcontroller first. The following figure is an excerpt from the Atmel AVR ATMega328P datasheet:

![A2_1](img/Mcu-unglitched.png)

Rather than loading each instruction from FLASH and performing the entire execution, the system has a pipeline to speed up the execution process. This means that an instruction is being decoded while the next one is being retrieved, as the following diagram shows:

![A2_2](img/Clock-normal.png)

But if we modify the clock, we could have a situation where the system doesn't have enough time to actually perform an instruction. Consider the following, where Execute #1 is effectively skipped. Before the system has time to actually execute it another clock edge comes, causing the microcontroller to start execution of the next instruction:

![A2_3](img/Clock-glitched.png)

This causes the microcontroller to skip an instruction. Such attacks can be immensely powerful in practice. Consider for example the following code from `linux-util-2.24`:

```C
/*
 *   auth.c -- PAM authorization code, common between chsh and chfn
 *   (c) 2012 by Cody Maloney <cmaloney@theoreticalchaos.com>
 *
 *   this program is free software.  you can redistribute it and
 *   modify it under the terms of the gnu general public license.
 *   there is no warranty.
 *
 */

#include "auth.h"
#include "pamfail.h"

int auth_pam(const char *service_name, uid_t uid, const char *username)
{
    if (uid != 0) {
        pam_handle_t *pamh = NULL;
        struct pam_conv conv = { misc_conv, NULL };
        int retcode;

        retcode = pam_start(service_name, username, &conv, &pamh);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_authenticate(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_acct_mgmt(pamh, 0);
        if (retcode == PAM_NEW_AUTHTOK_REQD)
            retcode =
                pam_chauthtok(pamh, PAM_CHANGE_EXPIRED_AUTHTOK);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_setcred(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        pam_end(pamh, 0);
        /* no need to establish a session; this isn't a
         * session-oriented activity...  */
    }
    return TRUE;
}
```

This is the login code for the Linux OS. Note that if we could skip the check of `if (uid != 0)` and simply branch to the end, we could avoid having to enter a password. This is the power of glitch attacks - not that we are breaking encryption, but simply bypassing the entire authentication module! 

### Glitch Hardware

The ChipWhisperer Glitch system uses the same synchronous methodology as its Side Channel Analysis (SCA) capture. A system clock (which can come from either the ChipWhisperer or the Device Under Test (DUT)) is used to generate the glitches. These glitches are then inserted back into the clock, although it's possible to use the glitches alone for other purposes (i.e. for voltage glitching, EM glitching).

The generation of glitches is done with two variable phase shift modules, configured as follows:

![A2_4](img/Glitchgen-phaseshift.png)

In CW-Husky there is one important difference: the phase shift 1 output is not inverted before it is ANDed with the phase shift 2 output.

The enable line is used to determine when glitches are inserted. Glitches can be inserted continuously (useful for development) or triggered by some event. The following figure shows how the glitch can be muxd to output to the Device Under Test (DUT).

![A2_5](img/Glitchgen-mux.png)

### Hardware Support: CW-Lite/Pro

The phase shift blocks use the Digital Clock Manager (DCM) blocks within the FPGA. These blocks have limited support for run-time configuration of parameters such as phase delay and frequency generation, and for maximum performance the configuration must be fixed at design time. The Xilinx-provided run-time adjustment can shift the phase only by about +/- 5nS in 30pS increments (exact values vary with operating conditions).

For most operating conditions this is insufficient - if attacking a target at 7.37MHz the clock cycle would have a period of 136nS. In order to provide a larger adjustment range, an advanced FPGA feature called Partial Reconfiguration (PR) is used. The PR system requires special partial bitstreams which contain modifications to the FPGA bitstream. These are stored as two files inside a "firmware" zip which contains both the FPGA bitstream along with a file called `glitchwidth.p` and a file called `glitchoffset.p`. If a lone bitstream is being loaded into the FPGA (i.e. not from the zip-file), the partial reconfiguration system is disabled, as loading incorrect partial reconfiguration files could damage the FPGA. This damage is mostly theoretical, more likely the FPGA will fail to function correctly.

If in the course of following this tutorial you find the FPGA appears to stop responding (i.e. certain features no longer work correctly), it could be the partial reconfiguration data is incorrect.

We'll look at how to interface with these features later in the tutorial.

### Hardware Support: CW-Husky

The clock-generation logic in Husky's 7-series FPGA is considerably different than the 6-series FPGAs used in CW-Lite/Pro. The DCM is gone and replaced by the much more powerful (and power hungry...) Mixed Mode Clock Manager (MMCM). In particular for our glitching application, MMCMs allow fine phase shift adjustments over an unlimited range, in steps as small as 15ps. And all this without having to dynamically reconfigure the FPGA bitfile! For this reason, the format for specifying the glitch offset and width is different from what it was for CW-Lite/Pro. Instead of specifiying a percentage of the source clock period, you now specify the actual number of phase shift steps. The duration of one phase shift step is 1/56 of the MMCM VCO clock period, which can itself be configured to be anyhwere in the range from 600 MHz to 1200 MHz (via `scope.clock.pll.update_fpga_vco()`).

While the MMCM is more powerful than the DCM with respect to its features, it also requires a lot more power. For this reason, the glitch generation circuitry is disabled by default and must be explicitly turned on. Fear not, Husky also uses Xilinx's XADC module to continuously monitor its temperature, and all MMCMs are automatically turned off at when the temperature starts getting too high, well below dangerous levels are reached (run `scope.XADC` to see all its statistics and settings).


In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../hardware/victims/firmware/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = "../../../hardware/victims/firmware/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=='SS_VER_2_1':
    target.reset_comms()

We'll probably crash the target a few times while we're trying some glitching. Create a function to reset the target:

In [ ]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.1)
        scope.io.pdic = "high_z"
        time.sleep(0.1)
        #Flush garbage too
        target.flush()
else:
    def reboot_flush():            
        scope.io.nrst = False
        time.sleep(0.05)
        scope.io.nrst = "high_z"
        time.sleep(0.05)
        #Flush garbage too
        target.flush()

## Communication

For this lab, we'll be introducing a new method: `target.simpleserial_read_witherrors()`. We're expecting a simpleserial response back; however, glitch will often cause the target to crash and return an invalid string. This method will handle all that for us. It'll also tell us whether the response was valid and what the error return code was. Use as follows:

In [ ]:
#Do glitch loop
target.simpleserial_write("g", bytearray([]))

val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

#print(bytearray(val['full_response'].encode('latin-1')))
print(val)

## Target Firmware

For this lab, our goal is to get the following code to preduce an incorrect result:

```C
uint8_t glitch_loop(uint8_t* in)
{
    volatile uint16_t i, j;
    volatile uint32_t cnt;
    cnt = 0;
    trigger_high();
    for(i=0; i<50; i++){
        for(j=0; j<50; j++){
            cnt++;
        }
    }
    trigger_low();
    simpleserial_put('r', 4, (uint8_t*)&cnt);
    return (cnt != 2500);
}
```

As you can see, we've got a simple loop. This is a really good place to start glitching for 2 reasons:

1. We've got a really long portion of time with a lot of instructions to glitch. In contrast, with the Linux example we're be trying to target a single instruction.

1. For some glitching scenarios, we're looking for a pretty specific glitch effect. In the Linux example, we might be banking on the glitch causing the target to skip an instruction instead of corrupting the comparison since that's a lot more likely to get us where we want in the code path. For this simple loop calculation, pretty much any malfunction will show up in the result.

## Glitch Module

All the settings/methods for the glitch module can be accessed under `scope.glitch`. As usual, documentation for the settings and methods can be accessed on [ReadtheDocs](https://chipwhisperer.readthedocs.io/en/latest/api.html) or with the python `help` command:

In [ ]:
help(scope.glitch)

#### We'll first go over settings that differ between the CW Husky and the CW Lite/Pro:
* clk_src

> The clock signal that the glitch DCM is using as input. Can be set to "target" or "clkgen" In this case, we'll be providing the clock to the target, so we'll want this set to "clkgen".

> On CW Husky, a separate PLL is used to clock the glitch module instead of the clkgen module. The equivalent setting here for "clkgen" is "pll"
* offset

> Where in the output clock to place the glitch. Can be in the range `[-48.8, 48.8]`. Often, we'll want to try many offsets when trying to glitch a target.

> On CW Husky, the range will depend on frequency of the PLL used to drive the glitch module (settable which can be configured to be anyhwere in the range from 600 MHz to 1200 MHz via `scope.clock.pll.update_fpga_vco()`), but, when the glitch module is active, the range will be `[0, scope.glitch.phase_shift_steps]`.
* width

> How wide to make the glitch. Can be in the range `[-50, 50]`, though there is no reason to use widths < 0. Wider glitches more easily cause glitches, but are also more likely to crash the target, meaning we'll often want to try a range of widths when attacking a target.

> Like offset, the range will be `[0, scope.glitch.phase_shift_steps]`.

#### These settings, on the other hand, are the same between the Husky and the Lite/Pro:

* output

> The output produced by the glitch module. For clock glitching, clock_xor is often the most useful option, as this inverts the clock during the glitch.
* ext_offset

> The number of clock cycles after the trigger to put the glitch.
* repeat

> The number of clock cycles to repeat the glitch for. Higher values increase the number of instructions that can be glitched, but often increase the risk of crashing the target.

* trigger_src

> How to trigger the glitch. For this tutorial, we want to automatically trigger the glitch from the trigger pin only after arming the ChipWhipserer, so we'll use `ext_single`

In addition, we'll need to tell ChipWhipserer to use the glitch module's output as a clock source for the target by setting `scope.io.hs2 = "glitch"`. We'll also setup a large `repeat` to make glitching easier.

## CW Glitch Controller

To make creating a glitch loop easier, ChipWhisperer includes a glitch controller. We'll start of by initializing with with different potential results of the attack. You define these to be whatever you want, but often three groups are sufficient:

1. `"success"`, where our glitch had the desired effect
1. `"reset"`, where our glitch had an undesirable effect. Often, this effect is crashing or resetting the target, which is why we're calling it `"reset"`
1. `"normal"`, where you glitch didn't have a noticable effect.

We also need to tell it what glitch parameters we want to scan through, in this case width and offset:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "tries"])

One of the niceties of the glitch controller is that it can display our current settings. This will update in real time as we use the glitch controller!

In [ ]:
gc.display_stats()

We can also make a settings plot that can also update in realtime as well:

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

Here `plotdots` is a dictionary that specifies how you want to plot each group. In this case, we're plotting `"success"` as a green `+` (`"+g"`), `"reset"` as a red `x` (`"xr"`), and we won't be plotting glitch attempts where nothing abnormal happens (`None`)

This plot will auto update its bounds as points are added. If you want to specify the axis bounds, you can do so as follows:

```python
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=(-48, 48), y_bound=(-48, 48))
```

You can also select which parameters you want to use for x and y, either by index, or by its name:

```python
# will flip width and offset axes
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index=1, y_index=0)
# or
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="offset", y_index="width")

```

You can set ranges for each glitch setting:

In [ ]:
gc.set_range("width", -5, 5)
gc.set_range("offset", -5, 5)

Each setting moves from min to max based on the global step:

In [ ]:
gc.set_global_step([5.0, 2.5])

We can print out all the glitch settings to see how this looks:

In [ ]:
for glitch_setting in gc.glitch_values():
    print("offset: {:4.1f}; width: {:4.1f}".format(glitch_setting[1], glitch_setting[0]))

You can tell the glitch controller when you've reached a particular result state like so:

In [ ]:
#gc.add("reset", (scope.glitch.width, scope.glitch.offset)) or simply gc.add("reset")
#gc.add("success", (scope.glitch.width, scope.glitch.offset)) or simply gc.add("success")

As of ChipWhisperer 5.7, you can skip the glitch width and glitch offset parameters. In this case, the glitch controller will use its internal values for the coordinates. Note that due to rounding, this will usually be a bit different from the actual hardware value on the Lite/Pro; however, the values will still correspond to the correct settings on your ChipWhisperer.

For CW-Husky, we must first explicitly turn on the glitch circuitry (it is off by default for power savings):

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True

We'll start off with the following settings. It's usually best to use "clock_xor" with clock glitching, which will insert a glitch if the clock is high or the clock is low.

In [ ]:
#Basic setup
# set glitch clock
if scope._is_husky:
    scope.glitch.clk_src = "pll"
else:
    scope.glitch.clk_src = "clkgen" 

scope.glitch.output = "clock_xor" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

scope.io.hs2 = "glitch"  # output glitch_out on the clock line
print(scope.glitch)

These settings are often a good starting point for all clock glitching, so, new with ChipWhisperer 5.7, we've got a method that sets all of this up for you:

In [ ]:
#scope.cglitch_setup()

You should have all you need to construct your glitch loop. We'll get you started, but the rest is up to you! Also, some stuff to keep in mind:

* You'll need to detect crashes, successful glitches, and normal returns from the target. Don't be afraid to experiment with the loop: you can always restart it and rerun the code.
* You can cover a larger set of glitch settings by starting with large glitch controller steps to get idea where some interesting locations are, then repeating the glitch loop with small steps in interesting areas. Where there's one successful glitch, there's probably more!
* You can speed up your glitch campaign substantially by only plotting crashes and successes, since they're typically much rarer than normal behaviour in the target
* On CW-Husky, glitch offset and width are specified in number of phase shift steps, whereas on CW-Lite/Pro, they are specified in percentage of clock period. The code provided below sets appropriate starting ranges for each case. Run `help(scope.glitch)` to understand this better.

In [ ]:
import chipwhisperer.common.results.glitch as glitch
from tqdm.notebook import trange
import struct

scope.glitch.ext_offset = 8

# width and offset numbers have a very different meaning for Husky vs Lite/Pro;
# see help(scope.glitch) for details
num_tries = 1
gc.set_range("tries", 1, num_tries)
if scope._is_husky:
    gc.set_range("width", 0, scope.glitch.phase_shift_steps)
    gc.set_range("offset", 0, scope.glitch.phase_shift_steps)
    gc.set_global_step([400, 200, 100])
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True
else:
    gc.set_range("width", 0, 48)
    gc.set_range("offset", -48, 48)
    gc.set_global_step([8, 4, 2, 1])

scope.glitch.repeat = 10
gc.set_step("tries", 1)
scope.adc.timeout = 0.1

reboot_flush()

for glitch_setting in gc.glitch_values():
    
    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]

    scope.arm()
    
    target.simpleserial_write("g", bytearray([]))
    
    ret = scope.capture()
    
    val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
    
    # ###################
    # Add your code here
    # ###################
    
    if ret: #here the trigger never went high - sometimes the target is still crashed from a previous glitch
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        if invalid_response: # change this to detect an invalid response
            gc.add("reset")
        else:
            # gcnt is the loop counter
            gcnt = struct.unpack("<I", val['payload'])[0]
            
            if gcnt == ???: #normal response
                gc.add("normal")
            else: #glitch!!!
                gc.add("success")

### Results

In addition to plotting, the glitch controller also has the capability to return results as a list that groups paramters and results. These results give both the number of each result, as well as the rate of each result:

In [ ]:
gc.calc()

You can also get results back with some parameters ignored. Results from parameters that now match will be grouped. This is particularly useful with something like the `"tries"` parameter, as you don't typically care whether a glitch was successful on your first, second, or third attempt:

In [ ]:
results = gc.calc(ignore_params="tries")
results

Finally, `calc()` can also sort by different results. A common use for this is to sort by success rate:

In [ ]:
results = gc.calc(ignore_params="tries", sort="success_rate")
results

### Plotting Glitch Results

We can replot our glitch map using the `plot_2d()` method. Settings are similar to `glitch_plot()`. If `plotdots` are not specified, the same ones as the `glitch_plot()` will be used.

In [ ]:
#gc.plot_2d(plotdots={"success":"+g", "reset":"xr", "normal":None})

Make sure you write down those glitch settings, since we'll be using for the rest of the glitching labs! In fact, we'll be using a lot of the general code structure here for the rest of the labs, with the only big changes being:

### Repeat

This lab used a pretty large repeat value. Like the name suggests, this setting controls how many times the glitch is repeated (i.e. a repeat value of 5 will place glitches in 5 consecutive clock cycles). Consider that each glitch inserted has a chance to both cause a glitch or crash the device. This was pretty advantageous for this lab since we had a lot of different spots we wanted to place a glitch - using a high repeat value increased our chance for a crash, but also increased our chance for a successful glitch. For an attack where we're targeting a single instruction, we don't really increase our glitch chance at all, but still have the increased crash risk. Worse yet, a successful glitch in a wrong spot may also cause a crash! It is for that reason that it's often better to use a low repeat value when targeting a single instruction.

### Ext Offset

The ext offset setting controls a delay between the trigger firing and the glitch being inserted. Like repeat, it's based on whole clock cycles, meaning an ext offset of 10 will insert a glitch 10 cycles after the trigger fires. We didn't have to worry about this setting for this lab since the large repeat value was able to take us into the area we wanted. This won't be true for many applications, where you'll have to try glitches at a large variety of ext_offsets.

### Success, Reset, and Normal

These three result states are usually enough to describe most glitch results. What constitues a success, however, will change based on what firmware you're attacking. For example, if we were attacking the Linux authentication, we might base success on a check to see whether or not we're root.

In [ ]:
scope.dis()
target.dis()

## Task 2

**SUMMARY:** *In the previous lab, we learned how clock glitching can be used to cause a target to behave unexpectedly. In this lab, we'll look at a slightly more realistic example - glitching past a password check*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching

## Firmware

We've already seen how we can insert clock gliches to mess up a calculation that a target is trying to make. While this has many applications, some which will be covered in Fault_201, let's take a look at something a little closer to our original example of glitch vulnerable code: a password check. No need to change out firmware here, we're still using the simpleserial-glitch project (though we'll go through all the build stuff again).

The code is as follows for the password check:

```C
uint8_t password(uint8_t* pw)
{
    char passwd[] = "touch";
    char passok = 1;
    int cnt;

    trigger_high();

    //Simple test - doesn't check for too-long password!
    for(cnt = 0; cnt < 5; cnt++){
        if (pw[cnt] != passwd[cnt]){
            passok = 0;
        }
    }
    
    trigger_low();
    
    simpleserial_put('r', 1, (uint8_t*)&passok);
    return passok;
}
```

There's really nothing out of the ordinary here - it's just a simple password check. We can communicate with it using the `'p'` command.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../hardware/victims/firmware/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../../hardware/victims/firmware/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

In [ ]:
def reboot_flush():
    reset_target(scope)
    #Flush garbage too
    target.flush()

If we send a wrong password:

In [ ]:
#Do glitch loop
reboot_flush()pw = bytearray([0x00]*5)
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

We get a response of zero. But if we send the correct password:

In [ ]:
#Do glitch loop
reboot_flush()pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68]) # correct password ASCII representation
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

We get a 1 back. Set the glitch up as in the previous part:

In [ ]:
scope.cglitch_setup()

Update the code below to also add an ext offset parameter:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

Like before, we'll plot our settings. The plot is 2D, so you'll need to make a decision about what to plot. In this case, we'll plot offset and ext_offset, but pick whichever you like:

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="offset")

And make a glitch loop. Make sure you use some successful settings that you found in the previous lab, since it will make this one much shorter!

One change you probably want to make is to add a scan for ext_offset. The number of places we can insert a successful glitch has gone way down. Doing this will also be very important for future labs.

In [ ]:
from tqdm.notebook import tqdm
import re
import struct
gc.set_global_step(step)

# replace these with your settings from the last part
# The example width/offset settings shown here are for CW-Lite/Pro; width/offset are expressed differently for Husky (see Fault 1_1)
gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)

# you can leave this range for ext_offset
gc.set_range(???, 0, 100)
gc.set_step(???, 1)

# adjust if you want
step = 1

scope.glitch.repeat = 1
reboot_flush()

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = ???
    scope.glitch.width = ???
    scope.glitch.ext_offset = ???
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        print("Trigger still high!")
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()

    scope.arm()
    target.simpleserial_write('p', bytearray(???)) #send an incorrect password

    ret = scope.capture()

    val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50) #For loop check
    if ret:
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()

    else:
        if invalid_response: #fill in with invalid response detection
            gc.add("reset")
        else:
            if glitched_past_pw_check: # replace this
                gc.add("success")
                print(val['payload'])
                print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                print("🐙", end="")
            else:
                gc.add("normal")

For this lab, you may want two copies of your results; one for finding effective ext_offsets:

In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

And one for your width/offset settings:

In [ ]:
results = gc.calc(ignore_params=["ext_offset"], sort="success_rate")
results

In [ ]:
scope.dis()
target.dis()

## Task 3

**SUMMARY:** *In the previous lab, we learned how clock glitching can be used to get a microcontroller to skip a password check. In this lab, we'll see if we can glitch a more realistic target: a bootloader's command response.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching
* Understanding how compiler optimizations can cause devices to behave in strange ways

## The Situation

Now that we've got our feet wet with glitching, we're going to try something a bit more realistic: an "encrypted" bootloader (it's actually just rot-13, but we'll pretend it's unbreakable encryption), where we make as few assumptions as possible. Our goal will be to get that bootloader to decrypt the data and send it back to us. Here's what we know about the bootloader:

1. The `'p'` command is used to write encrypted firmware to the device. It takes in an encrypted ASCII-encoded string, terminated with a newline. Our first chunk of firmware is `"516261276720736265747267206762206f686c207a76797821"`.
1. It does *something* to it (presumably unencrypts it, authenticates it, etc. and writes it to memory)
1. It sends back an error code of `"r000000\n"`

Of immediate interest is that error code. That's the only time the bootloader communicates back with us, so attacking there is a good place to start. One thing that we'll assume is that we've got a trigger right before the error code is sent back to us. This is just a simple `trigger_high()` call, but we could also trigger on an IO line (better with the CW1200 Pro) or with a SAD trigger on a power trace (CW1200 Pro only). We've got a place to start, but let's see if we can learn more about the bootloader first.

We recommend using SimpleSerial V2 for this as, though the firmware doesn't use the simpleserial protocol, the faster baud rate will help speed up glitching.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER="SS_VER_2_1"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../hardware/victims/firmware/bootloader-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../../hardware/victims/firmware/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)

In [ ]:
cw.program_target(scope, prog, fw_path)

The first thing we'll do is some simple power analysis to see what the device is doing when it sends data back to us. Serial communication is pretty slow, so set the ChipWhisperer to capture around 24k samples with a "x1" ADC clock.

In [ ]:
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

Next, capture a power trace. The string `"p516261276720736265747267206762206f686c207a76797821\n"` will send the bootloader the first chunk of code and plot it. If you don't see the full serial message, you can increase `scope.adc.decimate`, which will throw out every nth ADC sample.

In [ ]:
scope.adc.timeout = 3
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")
ret = scope.capture()
if ret:
    print("Timeout")
trace = scope.get_last_trace()


cw.plot(trace)

It doesn't look like anything too crazy is going on here - it's probably just printing some characters in a loop. Some ideas:

* If we glitch at the beginning of the loop, we might be able to corrupt the loop length variable and get it to print some extra memory
* We might be able to corrupt the loop variable and get it to read past where it's supposed to

For SimpleSerial V2, this should be short enough that you can quickly loop through the entirety of the code. If your target isn't using SimpleSerial V2, you should instead select a range a bit (~1000 cycles) before the end of the loop. If this doesn't succeed, you can try going after the cycles at the beginning of the loop.

**HINT: The last part of the loop should be near the beginning of the last power spike.**

**HINT: If you're really stuck on where the serial print ends, you can find the time between the `trigger_high()` and `trigger_low()` call with `scope.adc.trig_count`.**

In [ ]:
glitch_spots = [i for i in range(???)]
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

In [ ]:
print(glitch_spots)

### Evaluating Success

Detecting whether our glitch was successful or not isn't quite as trivial as in the previous lab - we don't have a nice error return that the device calculates and sends back to us. One idea is that we can look for part of the string that we sent to the device: there isn't much time between us sending it and the error code being returned. With any luck the compiler will have placed both values close in memory.

Now the rest is up to you! Use what you learned in the previous lab to setup glitch settings and a glitch loop. Here's a few hints to make things easier:

1. Try to use a fairly small width and offset range since we'll need to scan ext_offset as well here. A total range of ~2-3 for each with 0.4 steps is a good range to aim for. These numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1.
1. Try looking for a part of the string we sent to the device to check for success.
1. You may want to forgo graphing or plot only successes/crashes if it makes things substantially slower - we're scanning a large range of glitch settings so we'll need all the speed we can get.

Set your glitch up here:

In [ ]:
scope.adc.timeout = 0.1

scope.cglitch_setup()

def my_print(text):
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

Again, we can use the glitch controller to make loop setup easier:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

In [ ]:
x_bound = (-48, 48)
y_bound = (glitch_spots[0], glitch_spots[-1])
if scope._is_husky:
    x_bound = gc.set_range("width", 3900, 4500)
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=x_bound, y_bound=y_bound,
               x_index="width", y_index="ext_offset")

In [ ]:
# replace with good glitch settings you've found
gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)

gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])
gc.set_range("tries", 1, 1)
gc.set_step("tries", 1)
gc.set_step("ext_offset", 1)
scope.glitch.repeat = 1

    
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if broken:
        break
    if scope.adc.state:
        #print("Timeout, trigger still high!")
        gc.add("reset")
        #Device is slow to boot?
        reboot_flush()
        
    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        time.sleep(0.05)
        output = target.read(timeout=2)
        if ??? in output:
            print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
            gc.add("success")
            broken = True 
            for __ in range(500):
                num_char = target.in_waiting()
                if num_char:
                    my_print(output)
                    output = target.read(timeout=50)
            time.sleep(1)
            break
        else:
            gc.add("normal")

## Diagnosing the Fault

As you can see by the output, the bootloader has suffered a pretty catastrophic failure! Not only has it spilled the secret, it's also dumped a whole bunch more memory. For a real bootloader, there's probably some pretty juicy stuff in there like encryption keys or previously decrypted firmware. Let's start by taking a look at the C source code that sends the error code back:

```C
trigger_high();

int i;
for(i = 0; i < ascii_idx; i++)
{
    putch(ascii_buffer[i]);
}
trigger_low();
state = IDLE;
```

Nothing really looks too unusual here. Before we take a look at the assembly and figure out what went wrong, let's try to make some guesses:

* Maybe the glitch corrupted the `ascii_idx` variable
    * The glitch happened near the end of the loop. It's unlikely the end of loop counter would be reloaded during the loop
* Maybe we skipped the last `i < ascii_idx` check
    * The glitch caused **a lot** of memory to be dumped. If we just skipped the last check it **should** only print an extra character
* i is a signed integer: maybe we corrupted it into being a really large negative number.

That last one seems to be our best theory, so let's go with that.

## The Answer

Let's check the assembly for our booloader. No need to decompile the binary or recompile to assembly, since there's also a listing file created as part of the build process (`*.lss`). This file also contains C, so it makes it easy to search (try something like the `trigger_high()` call). You might notice that instead of doing a `less than or equal` or `less than` comparison like was in our C code, the compiler has instead inserted a `not equal` comparison instead! This means our original guess may not have been correct, as our assumption about what would happen if the last `i < ascii_idx` was skipped doesn't hold. In fact, it's a lot more likely that the last check was skipped (or i was set to some large value) than flipping a particular bit.

This is actually a pretty unexpected change for the compiler to make, espcially since `less than`, `greater than`, and `not equal` are nearly identical instructions in terms of implementation and have both the same instruction size and speed. This showcases an important fact: the C code that you write is not directly translated to assembly. It needs to go through the compiler first, which may drastically change the intended logic of the program.

Now that we know what happened, let's look at some ways to fix it.

### 1. Volatile variables

C includes a keyword for variables called `volatile`, which indicates that the variable may change between accesses and therefore should not have optimizations applied to it. A typical use case for `volatile` is for peripheral registers on embedded devices. It would be really bad, for example, if you were trying to wait for an IO pin to go high in your code, but the compiler decided it would be faster to only check it only once and assume it doesn't change!

Try replacing `int i = 0;` before the print look with `volatile int i = 0;`, recompile, and check the listing file. Is there any other unexpected changes? What about if you consider the use case above (i.e. if `i` was a register instead of a loop variable)? Is there any way the attack might still work? If so, how might you mitigate this?

### 2. Unrolling the loop

Another potential way of solving this issue would be to manually unroll the loop. The message being printed by the bootloader is a constant length of 7 characters, so we could instead write:

```C
int i;
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
```

In fact, this is something the compiler might do on its own to optimize the code, since unrolling a loop like this is faster than the loop version. It's not a good idea to blindly rely on this, however, since the compiler could choose not to make this optimization as well and might change it between builds.

### 3. Checking for invalid characters

Another thing to consider is that the message from the bootloader only has a limited range of characters that it prints. We could instead construct a "safe print" function that only prints newlines, `'r'` and ASCII digits (i.e. `'0'` to `'9'`):

```C
int safe_print(char c)
{
    if ((c == '\n') ||
       ((c >= '0') && (c <= '9')) ||
       (c == 'r')) {
        putch(c);
        return 0;
    }
    return -1; //uh oh!
}
```

It we went this route, it would be a good idea to make the error return a separate buffer with a bunch of null characters at the end.

### 4. More generic methods

More generic ways of defending against glitch attacks (memory guards, for example) are also discussed in the training slides.

In [ ]:
scope.dis()
target.dis()

## Task 4

**SUMMARY:** *In the previous lab, we saw how glitching might be used to get a bootloader to dump its entire memory by glitching its response to a command.*

*In this lab, we'll again be targeting a bootloader. This time, however, we have a different objective. Instead of trying to read firmware, we'll be trying to bypass the target's authentication in order to upload our own firmware.*

**LEARNING OUTCOMES:**

* Using glitching to bypass authentication of a bootloader command
* Performing a CPA attack using only traces that bypassed authentication

**Prerequisites**
This lab requires you to do a CPA attack on the bootloader. If you haven't yet, it's recommended that you run through SCA101 before attempting this lab.

## Target Bootloader

Throughout SCA101 (and even SCA201), we've focused exclusively on unauthenticated encryption. In real systems, this is often a very bad idea. If you're interested, the following blog post outlines some basic attacks on unauthenticated encryption: https://cybergibbons.com/reverse-engineering-2/why-is-unauthenticated-encryption-insecure/.

For our bootloader example, it would be catastrophic if we were able to modify the resulting firmware. We could then, for example, inject malicious code to dump out the firmware or encryption key. To try to prevent this, our target appends a message authentication code to each encrypted frame in an [Encrypt-then-MAC scheme](https://en.wikipedia.org/wiki/Authenticated_encryption#Encrypt-then-MAC). To keep things simple, the "hash function" is just an 8-bit CRC. When calculating the MAC (Message Authentication Code), the ciphertext is appended to a 64-bit key. Note  that this scheme is **not** secure; however, it is simple and it lets us reuse code from other parts of ChipWhisperer. In total, the whole frame this looks like:

* 16 bytes of encrypted (AES128-CBC) firmware
* 1 byte MAC

The frame is then wrapped in our standard SimpleSerial packet we'll be using the 'p' command (or 0x01 on SSV2) to send 
a frame to the target.

To start, setup the hardware as usual:

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../hardware/victims/firmware/simpleserial-aes-bootloader
make PLATFORM=$1 CRYPTO_TARGET=TINYAES128C SS_VER=$2

In [ ]:
fw_path = "../../../hardware/victims/firmware/simpleserial-aes-bootloader/simpleserial-bootloader-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
target.reset_comms()

We can pull in the CRC calculation from the SimpleSerial2 class:

In [ ]:
calc_crc = cw.targets.SimpleSerial2._calc_crc

In [ ]:
print(calc_crc([0x00, 0x12]))

def bootloader_calc_crc(buf):
    mac_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6]
    crc_buf = bytearray(mac_key)
    crc_buf.extend(buf)
    return calc_crc(crc_buf)

In [ ]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

print(bootloader_calc_crc(text))

We'll probably crash the target a few times while we're trying some glitching. Create a function to reset the target:

In [ ]:
def reboot_flush():
    reset_target(scope)
    target.reset_comms()
    target.simpleserial_write(0x00, bytearray(range(16)))
    target.simpleserial_wait_ack()

We want to catch both the MAC calculation and the beginning of AES, so we'll set `adc.samples` to its max:

In [ ]:
scope.adc.samples = 24400
scope.adc.offset = 5000

## Communication

To start off, we need to inititalize the bootloader with an IV:

In [ ]:
target.simpleserial_write(0x00, bytearray(range(16)))
target.simpleserial_wait_ack()

The bootloader protects against rewriting the IV, so if you rerun the block above the target won't update the IV and it'll return `0x12` as an error code.

Let's start by seeing what the power traces for a correctly authenticated message:

In [ ]:
#Do glitch loop
key, text = ktp.next()
text.extend([bootloader_calc_crc(text)])
scope.arm()
target.simpleserial_write("p", text)
scope.capture()
val = target.simpleserial_wait_ack() # For loop check

#print(bytearray(val['full_response'].encode('latin-1')))
print(val)
cw.plot(scope.get_last_trace())

and an incorrectly authenticated one:

In [ ]:
#Do glitch loop
key, text = ktp.next()
text.extend([0x00])
scope.arm()
target.simpleserial_write("p", text)
scope.capture()
val = target.simpleserial_wait_ack()#For loop check

#print(bytearray(val['full_response'].encode('latin-1')))
print(val)
cw.plot(scope.get_last_trace())

As you can see, the target only decrypts our message if it's got the right MAC. If the target didn't tell us that our MAC was wrong, we could use to help this determine if we got our glitch or not.

Next, we need to setup our glitch. By this point, you should have some fairly reliable settings you can use:

In [ ]:
scope.cglitch_setup(default_setup=False) # make sure default_setup is off so we don't overwrite our adc settings
gc = glitch.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, bufferlen=int(100E6))

In [ ]:
gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)
gc.set_range("ext_offset", ???, ???)
gc.set_step("ext_offset", 1)

We might get some "successful" glitches here that don't actually bypass the authentication. Therefore, let's get say 10 or so potentially successful glitches then check them afterwards for a good offset.

In [ ]:
scope.adc.timeout = 0.1
reboot_flush()
target.reset_comms()
x = []
req_encs = 10
encs = 0

project = cw.create_project("projects/test_bootloader", overwrite=True)

for glitch_setting in gc.glitch_values():

    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    if encs > req_encs:
        break
        
    scope.glitch.width = glitch_setting[0]
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.ext_offset = glitch_setting[2]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        print("Trigger still high!")
        gc.add("reset")
        reboot_flush()
        #target.reset_comms()

    #Do glitch loop
    key, text = ktp.next()
    cpy_text = bytearray(text)
    text.extend([0x00])
    scope.arm()
    target.simpleserial_write("p", text)
    ret = scope.capture()
    val = target.simpleserial_read('e', 1, ack=False)
    #print(val)

    if ret: #here the trigger never went high - sometimes the target is stil crashed from a previous glitch
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        target.reset_comms()
    else:
        if ???: # change this to detect an invalid response
            gc.add("reset")
        else:
            # gcnt is the loop counter
            gcnt = val[0]

            if gcnt == ???:#normal response
                gc.add("normal")
            elif gcnt == ???: #glitch!!!
                gc.add("success")
                print(f"Loc: {scope.glitch.ext_offset}, Glitch number: {encs}")
                x.append(scope.get_last_trace())
                trace = cw.Trace(scope.get_last_trace(), cpy_text, key, key)
                project.traces.append(trace)
                encs += 1


Let's plot these glitches:

In [ ]:
plot = cw.plot(x[0])
for y in x[1:]:
    plot += cw.plot(y)
plot

Hopefully you got at least one glitch that looks like the authenticated version of the encryption. We were able to use the return message to narrow down our glitches, but these other glitches shows that power traces are still very useful here.

Now that we know where to insert a glitch to bypass the authentication, we could use one of those attacks on unauthenticated encryption. Instead, let's do a CPA attack. That way, we'll be able to insert as much code without the limitations of those other attacks. Start by finding an offset that bypasses the authentication:

In [ ]:
cw.plot(x[???])

Next, let's repeat the attack, with the objective this time being to gather enough encryptions to recover the key:

In [ ]:
%%time
scope.glitch.ext_offset = 2

gc.set_range("ext_offset", 0, 10)
#gc.set_global_step([1])
scope.glitch.repeat = 1

scope.adc.timeout = 0.1

reboot_flush()
target.reset_comms()
x = []
req_encs = 100
scope.adc.offset = 10000
encs = 0

gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)
gc.set_range("ext_offset", 0, ???) # effectively "tries"
gc.set_step("ext_offset", 1)

project = cw.create_project("projects/test_bootloader", overwrite=True)

for glitch_setting in gc.glitch_values():

    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    if encs > req_encs:
        break

    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    scope.glitch.width = glitch_setting[0]
    scope.glitch.offset = glitch_setting[1]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        print("Trigger still high!")
        gc.add("reset")
        reboot_flush()
        target.reset_comms()

    #Do glitch loop
    key, text = ktp.next()
    cpy_text = bytearray(text)
    text.extend([0x00])
    scope.arm()
    target.simpleserial_write("p", text)
    ret = scope.capture()
    val = target.simpleserial_read('e', 1, ack=False, timeout=50)
    #print(val)

    if ret: #here the trigger never went high - sometimes the target is stil crashed from a previous glitch
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        target.reset_comms()
    else:
        if ???: # change this to detect an invalid response
            gc.add("reset")
        else:
            # gcnt is the loop counter
            gcnt = val[0]

            if gcnt == ???:#normal response
                gc.add("normal")
            else: #glitch!!!
                gc.add("success")
                print(scope.glitch.ext_offset)
                print(encs)
                x.append(scope.get_last_trace())
                trace = cw.Trace(scope.get_last_trace(), cpy_text, key, key)
                print(scope.glitch.width, scope.glitch.offset)
                project.traces.append(trace)
                encs += 1

Let's do a sanity check to make sure these traces all have encryptions (this will take a while to plot so you might only want to plot a subset of the traces)

In [ ]:
project.save()

In [ ]:
plot = cw.plot([])
for y in x:
    plot *= cw.plot(y)
plot

AES decryption and encryption are actually very similar, except with all the forward function replaced with inverse functions. For example, instead of the SBox, we have the inverse SBox, instead of MixColumns we have inverse MixColumns, etc. As such our attack will use the inverse SBox instead of the regular SBox (though we'll use the alternate model so the correct key gets highlighted in the table):

In [ ]:
import chipwhisperer as cw
project = cw.open_project("projects/test_bootloader")

Depending on your target, you may need to resync your traces:

In [ ]:
import chipwhisperer.analyzer as cwa
resync_traces = cwa.preprocessing.ResyncSAD(project)
resync_traces.ref_trace = 0
resync_traces.target_window = (???, ???)
resync_traces.max_shift = ???
resync_analyzer = resync_traces.preprocess()

plot = cw.plot([])
for y in range(10):
    plot *= cw.plot(resync_analyzer.waves[y])
plot

In [ ]:
import chipwhisperer.analyzer as cwa
leak_model = cwa.leakage_models.inverse_sbox_output_alt
attack = cwa.cpa(???, leak_model)
results = attack.run(cwa.get_jupyter_callback(attack))

This won't actually recover our usual encryption key. Instead we've got the last round key. We can use ChipWhisperer to convert it back to the original key:

In [ ]:
bytearray(cwa.aes_funcs.key_schedule_rounds(results.key_guess(), 10, 0))

And we can see it's our usual 0x2b7e... key.

## Injecting Malicious (Advanced)

Now that we can both bypass the MAC, as well as encrypt messages, try creating a piece of malicious code to dump firmware. Following commands will be useful:

1. `set_addr()` sets the address to write to
1. `go()` will run code pointed to by `set_addr()`

In [ ]:
scope.dis()
target.dis()